In [1]:
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import image as mpimg
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from tqdm import trange

In [2]:
! gdown https://avatars.githubusercontent.com/u/6183533?v=4 -O asharifiz.png

'gdown' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
class Layer:
    def __init__(self):
        self.inp = None
        self.out = None
    
    def __call__(self, inp:np.ndarray) -> np.ndarray:
        return self.forward(inp)
    
    def forward(self,inp:np.ndarray)->np.ndarray:
        raise NotImplementedError
    def backward(self,up_grad:np.ndarray)-> np.ndarray:
        raise NotImplementedError
    
    def step(self,lr:float) -> None:
        pass
    
    
class Linear(Layer):
    def __init__(self,in_dim:int,out_dim:int):
        super().__init__()
        
        self.w = 0.1*np.random.randn(in_dim,out_dim)
        self.b = np.zeros((1,out_dim))
        self.dw = np.zeros_like(self.w)
        self.db = np.zeros_like(self.b)
        
        
    def forward(self,inp:np.ndarray)-> np.ndarray:
        
        self.inp = inp
        self.out = np.dot(inp,self.w)+self.b
        return self.out
    
    def backward(self, up_grad:np.ndarray)->np.ndarray:
        
        self.dw = np.dot(self.inp.T,up_grad)
        self.db = np.sum(up_grad,axis=0,keepdims=True)
        
        down_grad = np.dot(up_grad,self.w.T)
        return down_grad
    
    def step(self,lr:float)->None:
        self.w -= lr*self.dw
        self.b -= lr*self.db
        

class ReLU(Layer):
    def forward(self, inp:np.ndarray)->np.ndarray:
        self.inp = inp
        self.out = np.maximum(0,inp)
        return self.out
    
    def backward(self, up_grad:np.ndarray)->np.ndarray:
        down_grad = up_grad*(self.inp>0)
        return down_grad
    
    

class Softmax(Layer):
    def forward(self, inp:np.ndarray)->np.ndarray:
        
        exp_values = np.exp(inp-np.max(inp,axis=1,keepdims=True))
        self.out = exp_values/np.sum(exp_values,axis=1,keepdims=True)
        return self.out
    
    
    def backward(self, up_grad:np.ndarray)-> np.ndarray:
        down_grad = np.empty_like(up_grad)
        for i in range(up_grad.shape[0]):
            single_output = self.out[i].reshape(-1,1)
            jacobian = np.diagflat(single_output)-np.dot(single_output,single_output.T)
            down_grad[i] = np.dot(jacobian,up_grad[i])
        return down_grad
    

class Loss:
    def __init__(self):
        self.prediction = None
        self.target = None
        self.loss = None
        
    def __call__(self,prediction:np.ndarray,target:np.ndarray)->float:
        return self.forward(prediction,target)
    def forward(self,prediction:np.ndarray,target:np.ndarray)->float:
        raise NotImplementedError
    
    def backward(self)-> np.ndarray:
        raise NotImplementedError
    
class CrossEntropy(Loss):
    def forward(self, prediction:np.ndarray, target:np.ndarray)->float:
        self.prediction = prediction
        self.target = target
        
        clipped_pred = np.clip(prediction,1e-12,1.0)
        self.loss = -np.mean(np.sum(target*np.log(clipped_pred),axis=1))
        return self.loss
    
    def backward(self)->np.ndarray:
        grad = -self.target/self.prediction/self.target.shape[0]
        return grad
    
    
class CNN:
    def __init__(self,layers:list[Layer],loss_fn:Loss,lr:float)->None:
        self.layers = layers
        self.loss_fn = loss_fn
        self.lr = lr
        
    def __call__(self, inp:np.ndarray)->np.ndarray:
        
        return self.forward(inp)
    
    def forward(self,inp:np.ndarray)->np.ndarray:
        
        for layer in self.layers:
            inp = layer.forward(inp)
        return inp
    
    def loss(self,prediction:np.ndarray,target:np.ndarray) ->float:
        
        return self.loss_fn(prediction,target)
    
    
    def backward(self)->None:
        up_grad = self.loss_fn.backward()
        for layer in reversed(self.layers):
            up_grad = layer.backward(up_grad)
    def update(self)->None:
        
        for layer in self.layers:
            layer.step(self.lr)
            
    def train(self,x_train:np.ndarray,y_train:np.ndarray,epochs:int,batch_size:int)->np.ndarray:
        
        losses,accuracies = np.empty(epochs),np.empty(epochs)
        for epoch in (pbar:=target(epochs)):
            running_loss = 0.0
            correct = 0
            for i in range(0,len(x_train),batch_size):
                x_batch = x_train[i:i+batch_size]
                y_batch = y_train[i:i+batch_size]
                
                prediction = self.forward(x_batch)
                
                running_loss += self.loss(prediction,y_batch)*batch_size
                
                correct += np.sum(np.argmax(prediction,axis=1)==np.argmax(y_batch,axis=1))
                
                
                self.backward()
                self.update()
                
            
            running_loss /= len(x_train)
            accuracy = 100*correct/len(x_train)
            pbar.set_description(f'Loss:{running_loss:.3f}| Accuracy:{accuracies:.2f}%')
            losses[epoch] = running_loss
            accuracies[epoch] = accuracy
        return losses,accuracies


In [4]:
def plot_conv(convolutions=None,img_src = 'asharifiz.png',seqentail=False):
    img = mpimg.imread(img_src)
    cols = len(convolutions) + 1 if convolutions is not None else 1
    fig,axes = plt.subplot(1,cols,figsize=(15,15*cols))
    
    axes[0].imshow(img)
    axes[0].axis('off')
    axes[0].set_title('Original')
    
    
    for i ,conv in enumerate(convolutions):
        x = np.transpose(img,axis=(0))
        out = conv(x).squeeze()
        out = np.tranpose(out,(1,2,0))
        axes[i+1].imshow(out)
        axes[i+1].axis('off')
        axes[i+1].set_title(f'Filter {i+1}')
        if seqentail:
            img = out

        plt.show()
        
def plot_results(losses,accuracies):
    
    plt.figure(figsize=(18,6))
    plt.subplot(1,2,1)
    plt.plot(range(1,len(losses)+1),accuracies)
    plt.title('Training Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.grid(True)
    plt.show()
    

def plot_confusion_matrix(y_true,y_pred,class_names,kept_classes):
    dim = len(kept_classes)
    labels = [class_names[i] for i in kept_classes]
    
    conf_mat = confusion_matrix(y_true,y_pred)
    norm_conf_mat = conf_mat/np.sum(conf_mat,axis=1)
    
    fig,ax = plt.Subplot()
    plt.imshow(norm_conf_mat)
    plt.title('confusion_matrix')
    plt.xlabel('Prediction')
    plt.ylabel('Labels')
    plt.xticks(range(dim),labels,rotation=45)
    plt.yticks(range(dim),labels)
    plt.colorbar()
    
    for i in range(dim):
        for j in range(dim):
            c = conf_mat[j,i]
            color = 'black' if c>500 else 'white'
            ax.text(i,j,str(int(c)),va='center',ha='center',color=color)
    plt.show()
    
    
def get_data(filter_classes):
    fashion_mnist = fetch_openml('Fashion_MNIST',parser='auto')
    x,y = fashion_mnist['data'],fashion_mnist['target'].astype(int)
    
    filter_indices = np.isin(y,filter_classes)
    x,y = x[filter_indices].to_numpy(),y[filter_indices]
    
    x = ((x/255.)-.5)*2
    removed_class_count = 0
    for i in range(10):
        if i in filter_classes and removed_class_count != 0:
            y[y==i] = i -removed_class_count
        elif i not in filter_classes:
            removed_class_count += 1
    return train_test_split(x,y,test_size=10_000)

def onehot_encoder(y,num_labels):
    one_hot = np.zeros(shape=(y.size,num_labels),dtype=int)
    one_hot[np.arange(y.size),y] = 1
    return one_hot



In [ ]:
class SimpleConv2D(Layer):
    def __init__(self,in_channels:int,out_channels:int,kernel_size:int):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        
        
        self.w = 0.1*np.random.randn(out_channels,in_channels,kernel_size,kernel_size)
        
    def forward(self, inp:np.array)->np.ndarray:
        self.inp = inp
        batch_size ,in_channels ,height,width = inp.shape
        assert in_channels == self.in_channels,'Input Channels must match'
        
        out_height = height - self.kernel_size+1
        out_width = width-self.kernel_size+1
        self.out = np.zeros((batch_size,self.out_channels,out_height,out_width))
        
        
        for i in range(out_height):
            for j in range(out_width):
                region = inp[:,:,i:i+self.kernel_sizem,j:j+self.kernel_size]
                self.out[:,:,i,j] = np.tensordot(region,self.w,axes=([1,2,3],[1,2,3]))+self.b.T
                
        return self.out
    